In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# df_train = pd.read_csv("../data/raw/favorita-grocery-sales-forecasting/train.csv", parse_dates=["date"])

In [ ]:
chunks = []

for chunk in pd.read_csv(
    "../data/raw/favorita-grocery-sales-forecasting/train.csv",
    parse_dates=["date"],
    chunksize=1_000_000
):
    filtered = chunk[chunk["date"] >= "2017-01-01"]

    if len(filtered):
        chunks.append(filtered)

df_train = pd.concat(chunks, ignore_index=True)

In [ ]:
df_train.shape

In [ ]:
df_train.head()

In [ ]:
df_train.dtypes

# drop id

In [ ]:
# drop id
df_train = df_train.drop(["id"], axis = 1)

In [ ]:
df_train.head()

In [ ]:
df_train.tail()

# reduce precision

In [ ]:
df_train["store_nbr"] = df_train["store_nbr"].astype("int8")

df_train["item_nbr"] = df_train["item_nbr"].astype("int32")

df_train["unit_sales"] = df_train["unit_sales"].astype("float32")

# df_train["lag1"] = df_train["lag1"].astype("float32")

# df_train["lag7"] = df_train["lag7"].astype("float32")

# set 0 or 1 to onpromotion

In [ ]:
df_train["onpromotion"] = df_train["onpromotion"].map(
    {False: 0, True: 1}, 
    na_action="ignore"
)

# df_train["onpromotion"] = df_train["onpromotion"].map(
#     {False: 0, True: 1}, 
# ).fillna(-1)

# df_train["onpromotion"] = (
#             df_train["onpromotion"]
#             .map({True: 1, False: 0, 1: 1, 0: 0})
#             .fillna(self.fill_value)
#             .astype("int8")
#         )#

# df_train["onpromotion"] = df_train["onpromotion"].replace({False: 0, True: 1})

In [ ]:
df_train.head()

In [ ]:
df_train[df_train['onpromotion'].isnull()]

In [ ]:
df_train.isnull().sum()

In [ ]:
df_train.dtypes

In [ ]:
df_train["onpromotion"] = df_train["onpromotion"].astype("int8")

# sort columns

In [ ]:
df_train = df_train.sort_values(["store_nbr", "item_nbr", "date"]).set_index("date")

In [ ]:
# df_train.head()

# create lag

In [ ]:
# check groupby
# df_train.groupby(["store_nbr","item_nbr"]).head()

## lag 1

In [ ]:
df_train["lag1"] = (
    df_train.groupby(["store_nbr","item_nbr"])["unit_sales"]
      .shift(1)
)

In [ ]:
# df_train.head()

In [ ]:
# df_train.tail()

## lag 7

In [ ]:
# df_train["lag7"] = (
#     df_train.groupby(["store_nbr","item_nbr"])["unit_sales"]
#       .shift(7)
# )

In [ ]:
# df_train.head(10)

In [ ]:
# df_train.dtypes

In [ ]:
# df_items = pd.read_csv("../data/raw/favorita-grocery-sales-forecasting/items.csv")


In [ ]:
# df_stores = pd.read_csv("../data/raw/favorita-grocery-sales-forecasting/stores.csv")

# merge features

In [ ]:
df_stores = pd.read_csv("../data/raw/favorita-grocery-sales-forecasting/stores.csv")

In [ ]:
df_train = df_train.merge(
    df_stores,
    on="store_nbr",
    how="left"
)

In [ ]:
df_train.head()

In [ ]:
df_items = pd.read_csv("../data/raw/favorita-grocery-sales-forecasting/items.csv")

In [ ]:
df_train = df_train.merge(
    df_items,
    on="item_nbr",
    how="left"
)

In [ ]:
df_train.head()

In [ ]:
df_oil = pd.read_csv("../data/raw/favorita-grocery-sales-forecasting/oil.csv", parse_dates=["date"])

In [ ]:
df_train = df_train.merge(
    df_oil,
    on="date",
    how="left"
)

In [ ]:
df_train.head()

In [ ]:
df_train["dcoilwtico"].isnull().sum()

# create sample weight for perishable (1.25 for 1 & 1.0 for 0)

In [ ]:
sample_weight = 1 + 0.25 * df_train["perishable"]

# plot

In [ ]:
# # plot unit sales

# df_train["unit_sales"].plot()

In [ ]:
# df_train = df_train.reset_index()
# df_train["date"] = pd.to_datetime(df_train["date"])

# include year month day

In [ ]:
df_train["year"] = df_train["date"].dt.year
df_train["month"] = df_train["date"].dt.month
df_train["day"] = df_train["date"].dt.day
# df["day_of_week"] = df["date"].dt.dayofweek
# df["day_of_year"] = df["date"].dt.dayofyear
# df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

In [ ]:
df_train = df_train.drop(["date"], axis = 1)

In [ ]:
df_train.head()

In [ ]:
df_train.dtypes

# reduce precision

In [ ]:
df_train["year"] = df_train["year"].astype("int16")

df_train["month"] = df_train["month"].astype("int8")

df_train["day"] = df_train["day"].astype("int8")

df_train["class"] = df_train["class"].astype("int16")

df_train["perishable"] = df_train["perishable"].astype("int8")

df_train["cluster"] = df_train["cluster"].astype("int8")

df_train["dcoilwtico"] = df_train["dcoilwtico"].astype("float32")

In [ ]:
df_train.head()

In [ ]:
df_train.dtypes

In [ ]:
df_train.describe()

In [ ]:
len(df_train.columns)

In [ ]:
categorical_features = [
    "store_nbr",
    "item_nbr",
    "city",
    "state",
    "type",
    "family",
]

numeric_features = [
    # "lag1",
    # "lag7",
    "onpromotion",
    "year",
    "month",
    "day",
    "class",
    "cluster",
    "perishable",
    "dcoilwtico"
]

# set negative values to zero and convert to log (skew to Gaussian)

In [ ]:
# df_train["unit_sales"] = np.clip(df_train["unit_sales"], 0, None)

df_train["unit_sales"] = np.log1p(np.maximum(0, df_train['unit_sales']))

# build X train & Y train

In [ ]:
X_train = df_train.drop(["unit_sales"], axis=1)
y_train = df_train["unit_sales"]

print(X_train.shape)
print(y_train.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
# import lightgbm as lgb 
from xgboost import XGBRegressor

In [ ]:
numeric_transformer = SimpleImputer(strategy='mean')

In [ ]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
# X_train_transformed = preprocessor.fit_transform(X_train)

# print(type(X_train_transformed))
# print(X_train_transformed.shape)

# print(X_train_transformed[:, 0:-1].dtype)
# # print(X_train_transformed[:, 2].dtype)

In [ ]:
# num_processed = preprocessor.named_transformers_["num"].transform(
#     X_train[numeric_features]
# ).astype(np.float32)

# print(num_processed.dtype)
# print(num_processed.shape)

In [ ]:


# model = RandomForestRegressor(
#     n_estimators=100,
#     max_depth=10,
#     min_samples_leaf=10,
#     random_state=42,
#     n_jobs=-1
# )

In [ ]:
# model=lgb.LGBMRegressor(
#         n_estimators=1000, verbose=10, random_state=42, learning_rate=0.03
#     )

In [ ]:
model = XGBRegressor(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.3,
    subsample=0.8,
    colsample_bytree=0.8,

    # Important for large datasets
    tree_method="hist",

    objective="reg:squarederror",
    eval_metric="rmse",

    n_jobs=-1,
    random_state=42
)

In [ ]:
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

In [ ]:
model_pipeline.fit(X_train, y_train, 
    model__sample_weight=sample_weight)

In [ ]:
# model_pipeline_xgb = Pipeline(
#     steps=[
#         ("preprocessor", preprocessor),
#         ("model", model_xgb)
#     ]
# )

In [ ]:
# model_xgb.fit(
#     X_train,
#     y_train,
#     verbose=50
# )

In [ ]:
df_test = pd.read_csv("../data/raw/favorita-grocery-sales-forecasting/test.csv", parse_dates=["date"])

In [ ]:
df_test.head()

In [ ]:
df_test = df_test.drop(["id"], axis = 1)

In [ ]:
df_test["year"] = df_test["date"].dt.year
df_test["month"] = df_test["date"].dt.month
df_test["day"] = df_test["date"].dt.day

In [ ]:
df_test = df_test.merge(
    df_stores,
    on="store_nbr",
    how="left"
)

In [ ]:
df_test = df_test.merge(
    df_items,
    on="item_nbr",
    how="left"
)

In [ ]:
df_test = df_test.merge(
    df_oil,
    on="date",
    how="left"
)

In [ ]:
df_test = df_test.drop(["date"], axis = 1)

In [ ]:
df_test["onpromotion"] = df_test["onpromotion"].map(
    {False: 0, True: 1}, 
    na_action="ignore"
)

In [ ]:
df_test.dtypes

In [ ]:
df_test["store_nbr"] = df_test["store_nbr"].astype("int8")

df_test["item_nbr"] = df_test["item_nbr"].astype("int32")

df_test["year"] = df_test["year"].astype("int16")

df_test["month"] = df_test["month"].astype("int8")

df_test["day"] = df_test["day"].astype("int8")

df_test["onpromotion"] = df_test["onpromotion"].astype("int8")

df_test["class"] = df_test["class"].astype("int16")

df_test["perishable"] = df_test["perishable"].astype("int8")

df_test["cluster"] = df_test["cluster"].astype("int8")

df_train["dcoilwtico"] = df_train["dcoilwtico"].astype("float32")

In [ ]:
X_test = df_test
print(X_test.shape)

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
y_test = model_pipeline.predict(X_test)

In [ ]:
df_sample = pd.read_csv("../data/raw/favorita-grocery-sales-forecasting/sample_submission.csv")

In [ ]:
df_sample["unit_sales"] = y_test

In [ ]:
df_sample.head()

In [ ]:
# df_sample["unit_sales"] = np.clip(df_sample["unit_sales"], 0, None)

df_sample["unit_sales"] = np.maximum(0, df_sample["unit_sales"])

In [ ]:
df_sample["unit_sales"] = np.expm1(df_sample["unit_sales"])

In [ ]:
df_sample.to_csv("../data/raw/favorita-grocery-sales-forecasting/submission_xgboost_with_sample_weights_log_transform.csv", index=False)

In [ ]:
df_sample.dtypes